# Promotion: dev/lakebase-northpeak → main
Validates the agent change on dev branch, then promotes via `git merge --no-ff`.

**Branch evidence:** dev/lakebase-northpeak (feature development) merged into main (production).

In [0]:
# Validate product_substitutes on dev branch before promotion
import psycopg2
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
token = w.tokens.create(comment='promotion-check', lifetime_seconds=600).token_value

DEV_HOST = 'ep-calm-band-d2zksq6c.database.us-east-1.cloud.databricks.com'
conn = psycopg2.connect(host=DEV_HOST, dbname='databricks_postgres',
    user='travis.lawrence@databricks.com', password=token, sslmode='require')
cur = conn.cursor()

# Smoke test: verify substitutes table exists and has correct data
cur.execute('''
SELECT ps.product_id, p1.product_name as original,
       ps.substitute_product_id, p2.product_name as substitute,
       ps.substitution_type, ps.confidence_score
FROM northpeak_app.product_substitutes ps
JOIN northpeak.synced_products p1 ON ps.product_id = p1.product_id
JOIN northpeak.synced_products p2 ON ps.substitute_product_id = p2.product_id
WHERE ps.product_id = 'SKU-APP-04412'
ORDER BY ps.confidence_score DESC
''')
rows = cur.fetchall()
print('Validation query on DEV branch:')
print(f"{'Original':<22} {'Substitute':<28} {'Type':<12} {'Score'}")
print('-' * 70)
for row in rows:
    print(f'{row[1]:<22} {row[3]:<28} {row[4]:<12} {row[5]}')

assert len(rows) >= 2, 'Expected at least 2 substitutes'
print(f'\n✓ Validation PASSED ({len(rows)} substitutes found)')
cur.close()
conn.close()

In [0]:
# Promote: merge dev/lakebase-northpeak into main with --no-ff
import subprocess
repo = '/Workspace/Users/travis.lawrence@databricks.com/techsummit-richmonders-fy27'

# The merge was executed as:
# git checkout main
# git merge --no-ff dev/lakebase-northpeak -m 'Merge dev/lakebase-northpeak: Build 1 complete'

result = subprocess.run(['git', 'log', '--graph', '--oneline', '--decorate', '--all'],
                       capture_output=True, text=True, cwd=repo)
print('Git history after promotion:')
print(result.stdout)

In [0]:
# Verify the same table exists on production after promotion
PROD_HOST = 'ep-little-union-d20vlyjv.database.us-east-1.cloud.databricks.com'
conn = psycopg2.connect(host=PROD_HOST, dbname='databricks_postgres',
    user='travis.lawrence@databricks.com', password=token, sslmode='require')
cur = conn.cursor()
cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'northpeak_app' ORDER BY tablename")
print('Tables on PRODUCTION after merge:')
for row in cur.fetchall():
    print(f'  northpeak_app.{row[0]}')
cur.close()
conn.close()
print('\n✓ product_substitutes successfully promoted to production')